# 01 — Data Pipeline: Feast Feature Engineering

**Prerequisite:** Raw data must already be in S3 (run the data-download Job manifest first).

```bash
# One-time: download HuggingFace → MinIO (see manifests/data-download-job.yaml)
envsubst < manifests/data-download-job.yaml | oc apply -f -
```

## What this notebook does

1. **Pre-flight** — verifies raw data (reviews + metadata) exists in S3
2. **Materialize** — triggers `feast materialize` via Feast SDK (server-side Spark + RAPIDS)
3. **Verify** — queries Redis via `get_online_features` to confirm materialized features

## How it works

Feast `@batch_feature_view` UDFs define PySpark transformations that run inside the
**SparkComputeEngine** (RAPIDS GPU) during `feast materialize`:

```
S3 raw reviews + metadata → feast materialize (Spark + RAPIDS) → Redis
```

- **user_features**: per-user aggregates (avg rating, review count, tenure, etc.)
- **item_features**: per-item aggregates + metadata join (title, brand, category, price)

The notebook uses `FeatureStore(fs_yaml_file="feast-config/smartshop")` — the Feast SDK
sends the materialize request to the remote offline server, which runs SparkComputeEngine
server-side. No pod exec or terminal access needed.

**Runs from:** RHOAI Workbench (in-cluster)


In [ ]:
%pip install -q boto3 tabulate pandas pyarrow feast redis pyyaml s3fs
%pip install yamlmagic --index-url https://pypi.org/simple
%load_ext yamlmagic

## Parameters

Edit these to customize the data pipeline run.


In [ ]:
%%yaml parameters

# No notebook-specific parameters needed for this notebook.
# Materialization is an admin operation run on the Feast pod.
_placeholder: true


In [ ]:
from _config import *
globals().update(parameters)

validate()

---
## Pre-flight: Verify Raw Data in S3

Confirms that raw reviews and metadata are already loaded in MinIO.
If empty, run the data-download manifest first (see header).


In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print(f"Existing buckets: {buckets}")

resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=5)
raw_files = resp.get("Contents", [])
if raw_files:
    print(f"\n✓ Raw reviews: {len(raw_files)}+ files in smartshop-raw/raw/reviews/")
else:
    print("\n✗ No raw data — run the download manifest first:")
    print("  envsubst < manifests/data-download-job.yaml | oc apply -f -")
    raise RuntimeError("Raw data missing")

expected_cats = ["Electronics", "Books", "Home_and_Kitchen"]
meta_resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/metadata/", MaxKeys=50)
meta_keys = [o["Key"] for o in meta_resp.get("Contents", [])]
found_cats = [c for c in expected_cats if any(c in k for k in meta_keys)]
missing_cats = [c for c in expected_cats if c not in found_cats]

print(f"✓ Metadata: {len(meta_keys)} files in smartshop-raw/raw/metadata/")
print(f"  Categories with metadata: {found_cats}")
if missing_cats:
    print(f"  ⚠ Missing metadata for: {missing_cats}")
    print("    Re-run download job with METADATA_ONLY=true to fetch these")

### Detailed S3 Listing


In [ ]:
from tabulate import tabulate

print("=== Reviews ===\n")
paginator = s3.get_paginator("list_objects_v2")
total_size = 0
total_files = 0
rows = []

for page in paginator.paginate(Bucket="smartshop-raw", Prefix="raw/reviews/"):
    for obj in page.get("Contents", []):
        size_mb = obj["Size"] / (1024 * 1024)
        total_size += obj["Size"]
        total_files += 1
        if total_files <= 15:
            rows.append([obj["Key"], f"{size_mb:.1f} MB"])

try:
    print(tabulate(rows, headers=["Key", "Size"], tablefmt="simple"))
except:
    for r in rows:
        print(f"  {r[0]:60s} {r[1]}")

if total_files > 15:
    print(f"  ... and {total_files - 15} more files")
print(f"\nTotal: {total_files} files, {total_size / (1024**3):.2f} GB")

print("\n=== Metadata ===\n")
meta_files = 0
meta_size = 0
for page in paginator.paginate(Bucket="smartshop-raw", Prefix="raw/metadata/"):
    for obj in page.get("Contents", []):
        meta_files += 1
        meta_size += obj["Size"]
        if meta_files <= 10:
            print(f"  {obj['Key']:60s} {obj['Size'] / (1024*1024):.1f} MB")
print(f"\nTotal metadata: {meta_files} files, {meta_size / (1024**3):.2f} GB")

In [ ]:
import pandas as pd
import io

# Quick peek at the first review file
resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=1)
if resp.get("Contents"):
    first_key = resp["Contents"][0]["Key"]
    obj = s3.get_object(Bucket="smartshop-raw", Key=first_key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    print(f"Preview: {first_key}")
    print(f"Rows: {len(df):,} | Columns: {list(df.columns)}")
    print()
    display(df.head(3)) if hasattr(__builtins__, '__IPYTHON__') else print(df.head(3).to_string())
else:
    print("No review files found — download may have failed.")

## 0.2 S3 Bucket Overview


In [ ]:
import s3fs

fs = s3fs.S3FileSystem(
    key=AWS_KEY,
    secret=AWS_SECRET,
    client_kwargs={"endpoint_url": S3_ENDPOINT},
)

for bucket in ["smartshop-raw", "smartshop-features", "smartshop-models"]:
    try:
        files = fs.ls(bucket)
        print(f"\n{bucket}/ ({len(files)} top-level items)")
        for f in files[:8]:
            print(f"  {f}")
        if len(files) > 8:
            print(f"  ... and {len(files) - 8} more")
    except Exception:
        print(f"\n{bucket}/ (not found or empty)")


---
# Phase 1: Feast Feature Engineering + Materialization

Trigger `feast materialize` via the Feast SDK. The request is sent to the remote offline server
which runs the **SparkComputeEngine** (RAPIDS GPU) server-side:
1. Reads raw reviews + metadata from S3
2. Applies `@batch_feature_view` PySpark UDFs (groupBy, agg, join)
3. Writes computed features directly to Redis


In [ ]:
import sys, os
from feast import FeatureStore
from datetime import datetime, timezone

sys.path.insert(0, os.path.join(os.getcwd(), "feature_repo"))
from features import user, item, raw_reviews_source, user_features, item_features

store = FeatureStore(fs_yaml_file=FEAST_CLIENT_CONFIG)

print("--- feast apply ---")
store.apply([user, item, raw_reviews_source, user_features, item_features])

print("\nRegistered entities:")
for e in store.list_entities():
    print(f"  {e.name}")
print("Registered feature views:")
for fv in store.list_feature_views():
    print(f"  {fv.name} ({len(fv.features)} features)")
print("✓ apply complete")

---
# Phase 2: Verify Features in Redis

Query the Redis online store via `get_online_features()` to confirm materialized data.


In [ ]:
import redis, struct
import pandas as pd

print("Feast registry connected ✓")
for fv in store.list_feature_views():
    features = [f.name for f in fv.features] if hasattr(fv, "features") else []
    print(f"  {fv.name:25s} ({len(features)} features)")

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, password=REDIS_PASSWORD or None, decode_responses=False)
print(f"\nRedis keys: {r.dbsize():,}")


### User Features


In [ ]:
import pandas as pd

# Discover real user IDs from Redis — single SCAN call, parse Feast v3 binary key format
import struct

_, raw_keys = r.scan(cursor=0, match=b"*user_id*smartshop", count=20)
user_ids = []
for rk in raw_keys:
    raw = rk if isinstance(rk, bytes) else rk.encode()
    marker = b"user_id"
    pos = raw.find(marker)
    if pos < 0:
        continue
    # After entity name: \x02\x00\x00\x00 (type tag) + 4-byte little-endian length + value + "smartshop"
    offset = pos + len(marker) + 4  # skip type tag
    if offset + 4 > len(raw):
        continue
    val_len = struct.unpack("<I", raw[offset:offset + 4])[0]
    val = raw[offset + 4 : offset + 4 + val_len].decode("utf-8", errors="ignore")
    if val:
        user_ids.append(val)

sample_users = [{"user_id": uid} for uid in user_ids[:5]]
print(f"Querying {len(sample_users)} real user IDs: {[u['user_id'] for u in sample_users]}\n")

result = store.get_online_features(
    features=[
        "user_features:user_avg_rating",
        "user_features:user_review_count",
        "user_features:user_unique_items",
        "user_features:user_tenure_days",
    ],
    entity_rows=sample_users,
).to_dict()

df = pd.DataFrame(result)
print("User features from Redis:")
display(df) if hasattr(__builtins__, '__IPYTHON__') else print(df.to_string())

### Item Features


In [ ]:
# Discover real item IDs (ASINs) from Redis — scan enough keys for a good sample
_, raw_keys = r.scan(cursor=0, match=b"*item_id*smartshop", count=200)
item_ids = []
for rk in raw_keys:
    raw = rk if isinstance(rk, bytes) else rk.encode()
    marker = b"item_id"
    pos = raw.find(marker)
    if pos < 0:
        continue
    offset = pos + len(marker) + 4
    if offset + 4 > len(raw):
        continue
    val_len = struct.unpack("<I", raw[offset:offset + 4])[0]
    val = raw[offset + 4 : offset + 4 + val_len].decode("utf-8", errors="ignore")
    if val:
        item_ids.append(val)

sample_items = [{"item_id": iid} for iid in item_ids[:5]]
print(f"Querying {len(sample_items)} real item IDs (ASINs): {[i['item_id'] for i in sample_items]}\n")

result = store.get_online_features(
    features=[
        "item_features:item_title",
        "item_features:item_brand",
        "item_features:item_category",
        "item_features:item_avg_rating",
        "item_features:item_price",
        "item_features:item_review_count",
    ],
    entity_rows=sample_items,
).to_dict()

df = pd.DataFrame(result)
has_meta = df["item_title"].notna().sum()
print(f"Item features from Redis ({has_meta}/{len(df)} have product metadata):")
display(df) if hasattr(__builtins__, '__IPYTHON__') else print(df.to_string())
if has_meta == 0:
    print("\n⚠ All metadata columns are None — check that metadata was downloaded for all categories")

---
## Summary

| Phase | Step | What happened |
|-------|------|---------------|
| **0** | Pre-flight | Verified raw data in S3 (reviews + metadata for 3 categories) |
| **1** | Materialize | `store.materialize()` via Feast SDK → server-side Spark + RAPIDS GPU → Redis |
| **2** | Validation | `get_online_features()` confirmed user/item features + product metadata in Redis |

**Key RHOAI capabilities shown:**
- Feast `@batch_feature_view` with PySpark UDFs for feature engineering
- SparkComputeEngine with NVIDIA RAPIDS GPU acceleration
- Feast SDK triggers materialization from notebook — server handles Spark/GPU/Redis
- Product metadata enrichment (title, brand, category, price) via inline join

**Next:** Run `02_training.ipynb` to train the recommendation model and fine-tune the LLM.
